In [0]:
dbutils.widgets.dropdown(name="environment",defaultValue="dev",choices=["dev", "prod", "qa"],label="Select Environment")

In [0]:
env = dbutils.widgets.get("environment")
print(env)

In [0]:
brznTablName = f"saleslake_{env}.bronze_{env}.rawCustomer"

silverTableName = f"saleslake_{env}.silver_{env}.cleanedCustomer"

srcFileLoc = f"/Volumes/saleslake_{env}/bronze_{env}/vol_saleslake_src_files_{env}/daily_customer/"

print(brznTablName)
print(silverTableName)
print(srcFileLoc)

In [0]:

spark.sql(f"""
INSERT INTO {silverTableName} 
SELECT DISTINCT   
     CAST(TRIM(customer_id) AS INTEGER) as customer_id ,
     UPPER(TRIM(customer_name)) as customer_name,
     UPPER(TRIM(email)) as email,
    UPPER(TRIM(phone)) as phone,
     UPPER(TRIM(address)) as address,
     UPPER(TRIM(city)) as city,
     UPPER(TRIM(state)) as state,
     UPPER(TRIM(country)) as country,
     UPPER(TRIM(zip_code)) as zip_code,
     UPPER(TRIM(segment)) as segment,
     CURRENT_TIMESTAMP() as ingest_ts
 FROM {brznTablName}
 WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                     FROM {silverTableName}
                    )
 ORDER BY CAST(TRIM(customer_id) AS INTEGER)
""")